# [9.3] Emergent Misalignment Detection - Solutions

This notebook runs the reference implementation for the benign proxy-drift section. The source implementation lives in `solutions.py`; the cells below keep the same conceptual arc as the exercise notebook and show the expected outputs.

<details>
<summary>Expected output</summary>

All visible tests should pass, the smoke contract should print the toy proxy reports, and the committed CUDA signature table should match the report.
</details>

<details>
<summary>Help - what should I focus on?</summary>

The main lesson is the evidence ladder: safe proxy taxonomy, held-out hidden-state detection, signed behavior alignment, negative controls, narrow mitigation, and explicit non-claims.
</details>


In [1]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part3_emergent_misalignment_detection"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_emergent_misalignment_detection.solutions as solutions
import part3_emergent_misalignment_detection.tests as tests
import part3_emergent_misalignment_detection.utils as utils


## Visible Unit Tests

<details>
<summary>Expected output</summary>

Each visible test prints `All tests in ... passed!`. The invalid-input tests are part of the value: they stop empty tensors, non-finite evidence, one-class logits, constant correlations, and bad hidden-state direction fits from passing silently.
</details>

<details>
<summary>Help - why so many small tests?</summary>

ARENA-style notebooks catch mistakes at the function where they happen. In this section, most dangerous errors are not syntax errors; they are weak evidence checks that produce plausible-looking numbers.
</details>


In [2]:
tests.test_proxy_kinds_are_explicit_safe_categories(
    solutions.proxy_kinds_smoke_test,
)
tests.test_drift_detector_report_scores_heldout_logits(
    solutions.drift_detector_report,
)
tests.test_crosscoder_alignment_uses_pearson_correlation(
    solutions.crosscoder_drift_alignment_report,
)
tests.test_mitigation_report_bounds_capability_loss(
    solutions.drift_mitigation_report,
)
tests.test_early_warning_report_compares_detection_steps(
    solutions.early_warning_report,
)
tests.test_detector_smoke_test(solutions.detector_smoke_test)
tests.test_crosscoder_smoke_test(solutions.crosscoder_smoke_test)
tests.test_mitigation_smoke_test(solutions.mitigation_smoke_test)
tests.test_early_warning_smoke_test(solutions.early_warning_smoke_test)
tests.test_pythia_direction_fit_rejects_invalid_internal_evidence()
tests.test_behavior_proxy_tokens_must_be_single_tokens()


All tests in `test_proxy_kinds_are_explicit_safe_categories` passed!
All tests in `test_drift_detector_report_scores_heldout_logits` passed!
All tests in `test_crosscoder_alignment_uses_pearson_correlation` passed!
All tests in `test_mitigation_report_bounds_capability_loss` passed!
All tests in `test_early_warning_report_compares_detection_steps` passed!
All tests in `test_detector_smoke_test` passed!
All tests in `test_crosscoder_smoke_test` passed!
All tests in `test_mitigation_smoke_test` passed!
All tests in `test_early_warning_smoke_test` passed!
All tests in `test_pythia_direction_fit_rejects_invalid_internal_evidence` passed!
All tests in `test_behavior_proxy_tokens_must_be_single_tokens` passed!


## Notebook Contract

<details>
<summary>Expected output</summary>

The smoke contract should include the five benign proxy kinds, a perfect toy held-out detector, positive feature-behavior alignment, passing mitigation, and earlier white-box timing.
</details>

<details>
<summary>Help - interpreting the toy contract</summary>

The toy contract proves the local report functions behave as intended. It is deliberately not the GT-3 evidence path; the Pythia report is checked separately.
</details>


In [3]:
contract = solutions.run_smoke_test(cpu=True)
utils.print_report("Drift detector", contract["detector"])
utils.print_report("Proxy feature alignment", contract["crosscoder"])
utils.print_report("Drift mitigation", contract["mitigation"])
utils.print_report("Early warning", contract["early_warning"])
tests.test_notebook_contract(solutions.run_smoke_test)


Drift detector
  detector_accuracy      : 1.0
  predicts_heldout_drift : True
Proxy feature alignment
  correlation                : 0.9994795322418213
  aligns_with_behavior_delta : True
Drift mitigation
  baseline_drift_score  : 0.75
  mitigated_drift_score : 0.3500000238418579
  drift_reduction       : 0.3999999761581421
  capability_loss       : 0.0350000262260437
  mitigation_passes     : True
Early warning
  white_box_detection_step  : 2
  black_box_detection_step  : 5
  white_box_catches_earlier : True
All tests in `test_notebook_contract` passed!


## Signature Result

The report-backed result is a Pythia-70M benign proxy-drift hidden-state preflight, not an emergent-misalignment reproduction.

<img src="../../instructions/assets/emergent_misalignment_signature_result.svg" width="860">

<details>
<summary>Interpreting the signature result</summary>

The detector works on held-out safe proxy prompts, aligns with a two-token behavior proxy, and fails the label-shuffle and random-direction controls. The mitigation result is a narrow LM-head readout check, not a deployment claim.
</details>


In [4]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

signature = {
    "preflight_passed": gpu["preflight_passed"],
    "model": gpu["model_name"],
    "hidden_state_shape": gpu["hidden_state_shape"],
    "detector_accuracy": gpu["detector_accuracy"],
    "drift_alignment_correlation": gpu["drift_alignment_correlation"],
    "label_shuffled_detector_accuracy": gpu["label_shuffled_detector_accuracy"],
    "random_direction_accuracy": gpu["random_direction_accuracy"],
    "mitigation_drift_delta_reduction": gpu["mitigation_drift_delta_reduction"],
    "generation_used": gpu["generation_used"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
utils.print_report("Pythia proxy-drift preflight", signature)
tests.test_committed_gpu_report_matches_proxy_drift_contract(report)


Pythia proxy-drift preflight
  preflight_passed                 : True
  model                            : EleutherAI/pythia-70m-deduped
  hidden_state_shape               : [24, 512]
  detector_accuracy                : 1.0
  drift_alignment_correlation      : 0.7360556721687317
  label_shuffled_detector_accuracy : 0.6666666865348816
  random_direction_accuracy        : 0.375
  mitigation_drift_delta_reduction : 1.0936462879180908
  generation_used                  : False
  peak_vram_gb                     : 0.3098287582397461
All tests in `test_committed_gpu_report_matches_proxy_drift_contract` passed!


## Limitations

This section supports a narrow safe preflight, not a broad safety conclusion.

- The proxy drift categories are benign policy-description changes, not harmful finetunes.
- The real-model path uses hidden states and logits only; it does not evaluate generated completions.
- The feature-alignment exercise is crosscoder-style, but no crosscoder is trained.
- The toy early-warning timing check is not a live checkpoint-series experiment.
- The mitigation check is an LM-head projection readout on hidden states, not a deployed intervention.

## Further Research

Replace the two-token behavior proxy with a safe held-out behavior suite, measure early-warning timing over checkpoint series, and compare this direction baseline with a trained crosscoder or model-diffing feature map.
